# Smart Meters: inspección inicial de datasets originales

Esta libreta inicia la exploración de los datasets originales descargados desde Kaggle para el proyecto de Smart Energy Analytics.

El objetivo de esta etapa es entender estructura, tamaño, granularidad, columnas clave y relaciones entre archivos antes de construir un nuevo modelo de clustering de comportamiento de consumo.

La inspección se hace de forma cuidadosa porque el dataset completo descomprimido pesa varios GB. En particular, los archivos de media hora completos no se cargan enteros; se revisan por muestra o por bloque.

# 1. Configuración general

Se definen rutas, librerías y opciones de visualización.

In [1]:
from pathlib import Path
import os
import warnings

os.environ.setdefault("MPLCONFIGDIR", str((Path("./.matplotlib_cache")).resolve()))
os.environ.setdefault("XDG_CACHE_HOME", str((Path("./.cache")).resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
ORIGINAL_BASE = DATA_DIR / "original" / "kagglehub" / "datasets" / "jeanmidev" / "smart-meters-in-london" / "versions" / "21"
OUTPUT_DIR = PROJECT_DIR / "output" / "inspeccion_originales"
IMAGES_DIR = OUTPUT_DIR / "images"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print("Directorio del proyecto:", PROJECT_DIR)
print("Base de datos originales:", ORIGINAL_BASE)
print("Existe base original:", ORIGINAL_BASE.exists())

Directorio del proyecto: /Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics
Base de datos originales: /Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21
Existe base original: True


# 2. Inventario de archivos descargados

Se listan archivos principales, tamaños y carpetas de bloques. Esto permite confirmar qué datasets están disponibles y cuáles deben tratarse con cuidado por tamaño.

In [2]:
def size_mb(path):
    return path.stat().st_size / (1024 ** 2)

def size_gb_dir(path):
    total = sum(p.stat().st_size for p in path.rglob("*") if p.is_file())
    return total / (1024 ** 3)

archivos_principales = {
    "daily_dataset.csv": ORIGINAL_BASE / "daily_dataset.csv",
    "informations_households.csv": ORIGINAL_BASE / "informations_households.csv",
    "weather_daily_darksky.csv": ORIGINAL_BASE / "weather_daily_darksky.csv",
    "acorn_details.csv": ORIGINAL_BASE / "acorn_details.csv",
}

carpetas_bloques = {
    "daily_blocks": ORIGINAL_BASE / "daily_dataset" / "daily_dataset",
    "hhblock_blocks": ORIGINAL_BASE / "hhblock_dataset" / "hhblock_dataset",
    "halfhourly_blocks": ORIGINAL_BASE / "halfhourly_dataset" / "halfhourly_dataset",
}

inventario = []
for nombre, path in archivos_principales.items():
    inventario.append({
        "tipo": "archivo",
        "nombre": nombre,
        "existe": path.exists(),
        "tamano_mb": round(size_mb(path), 2) if path.exists() else np.nan,
        "num_archivos": 1 if path.exists() else 0,
        "ruta": str(path)
    })

for nombre, path in carpetas_bloques.items():
    archivos = sorted(path.glob("block_*.csv")) if path.exists() else []
    inventario.append({
        "tipo": "carpeta",
        "nombre": nombre,
        "existe": path.exists(),
        "tamano_mb": round(sum(size_mb(p) for p in archivos), 2) if archivos else np.nan,
        "num_archivos": len(archivos),
        "ruta": str(path)
    })

df_inventario = pd.DataFrame(inventario)
print("Tamaño total aproximado de ORIGINAL_BASE:", round(size_gb_dir(ORIGINAL_BASE), 2), "GB")
display(df_inventario)

Tamaño total aproximado de ORIGINAL_BASE: 9.56 GB


,tipo,nombre,existe,tamano_mb,num_archivos,ruta
0,archivo,daily_dataset.csv,True,335.5000,1,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...
1,archivo,informations_households.csv,True,0.2200,1,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...
2,archivo,weather_daily_darksky.csv,True,0.3300,1,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...
3,archivo,acorn_details.csv,True,0.1200,1,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...
4,carpeta,daily_blocks,True,357.1300,112,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...
5,carpeta,hhblock_blocks,True,"1,593.8400",112,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...
6,carpeta,halfhourly_blocks,True,"7,501.3000",112,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...


# 3. Inspección rápida de esquemas

Se leen pocas filas de cada archivo clave para revisar nombres de columnas, tipos inferidos y ejemplos.

In [3]:
def leer_muestra_csv(path, nrows=5, encoding="latin1"):
    return pd.read_csv(path, nrows=nrows, encoding=encoding)

archivos_muestra = {
    **archivos_principales,
    "daily_block_0.csv": carpetas_bloques["daily_blocks"] / "block_0.csv",
    "hhblock_block_0.csv": carpetas_bloques["hhblock_blocks"] / "block_0.csv",
    "halfhourly_block_0.csv": carpetas_bloques["halfhourly_blocks"] / "block_0.csv",
}

for nombre, path in archivos_muestra.items():
    print("\n" + "=" * 80)
    print(nombre)
    print("Ruta:", path)
    print("Existe:", path.exists())
    if path.exists():
        df_tmp = leer_muestra_csv(path, nrows=5)
        print("Shape muestra:", df_tmp.shape)
        print("Columnas:", df_tmp.columns.tolist())
        display(df_tmp.head())


daily_dataset.csv
Ruta: /Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21/daily_dataset.csv
Existe: True
Shape muestra: (5, 9)
Columnas: ['LCLid', 'day', 'energy_median', 'energy_mean', 'energy_max', 'energy_count', 'energy_std', 'energy_sum', 'energy_min']


,LCLid,day,energy_median,energy_mean,energy_max,energy_count,energy_std,energy_sum,energy_min
0,MAC000131,2011-12-15,0.4850,0.4320,0.8680,22,0.2391,9.5050,0.0720
1,MAC000131,2011-12-16,0.1415,0.2962,1.1160,48,0.2815,14.2160,0.0310
2,MAC000131,2011-12-17,0.1015,0.1898,0.6850,48,0.1884,9.1110,0.0640
3,MAC000131,2011-12-18,0.1140,0.2190,0.6760,48,0.2029,10.5110,0.0650
4,MAC000131,2011-12-19,0.1910,0.3260,0.7880,48,0.2592,15.6470,0.0660



informations_households.csv
Ruta: /Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21/informations_households.csv
Existe: True
Shape muestra: (5, 5)
Columnas: ['LCLid', 'stdorToU', 'Acorn', 'Acorn_grouped', 'file']


,LCLid,stdorToU,Acorn,Acorn_grouped,file
0,MAC005492,ToU,ACORN-,ACORN-,block_0
1,MAC001074,ToU,ACORN-,ACORN-,block_0
2,MAC000002,Std,ACORN-A,Affluent,block_0
3,MAC003613,Std,ACORN-A,Affluent,block_0
4,MAC003597,Std,ACORN-A,Affluent,block_0



weather_daily_darksky.csv
Ruta: /Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21/weather_daily_darksky.csv
Existe: True
Shape muestra: (5, 32)
Columnas: ['temperatureMax', 'temperatureMaxTime', 'windBearing', 'icon', 'dewPoint', 'temperatureMinTime', 'cloudCover', 'windSpeed', 'pressure', 'apparentTemperatureMinTime', 'apparentTemperatureHigh', 'precipType', 'visibility', 'humidity', 'apparentTemperatureHighTime', 'apparentTemperatureLow', 'apparentTemperatureMax', 'uvIndex', 'time', 'sunsetTime', 'temperatureLow', 'temperatureMin', 'temperatureHigh', 'sunriseTime', 'temperatureHighTime', 'uvIndexTime', 'summary', 'temperatureLowTime', 'apparentTemperatureMin', 'apparentTemperatureMaxTime', 'apparentTemperatureLowTime', 'moonPhase']


...

,LCLid,day,energy_median,energy_mean,energy_max,energy_count,energy_std,energy_sum,energy_min
0,MAC000002,2012-10-12,0.1385,0.1543,0.8860,46,0.1960,7.0980,0.0000
1,MAC000002,2012-10-13,0.1800,0.2310,0.9330,48,0.1923,11.0870,0.0760
2,MAC000002,2012-10-14,0.1580,0.2755,1.0850,48,0.2746,13.2230,0.0700
3,MAC000002,2012-10-15,0.1310,0.2137,1.1640,48,0.2245,10.2570,0.0700
4,MAC000002,2012-10-16,0.1450,0.2035,0.9910,48,0.1841,9.7690,0.0870



hhblock_block_0.csv
Ruta: /Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21/hhblock_dataset/hhblock_dataset/block_0.csv
Existe: True
Shape muestra: (5, 50)
Columnas: ['LCLid', 'day', 'hh_0', 'hh_1', 'hh_2', 'hh_3', 'hh_4', 'hh_5', 'hh_6', 'hh_7', 'hh_8', 'hh_9', 'hh_10', 'hh_11', 'hh_12', 'hh_13', 'hh_14', 'hh_15', 'hh_16', 'hh_17', 'hh_18', 'hh_19', 'hh_20', 'hh_21', 'hh_22', 'hh_23', 'hh_24', 'hh_25', 'hh_26', 'hh_27', 'hh_28', 'hh_29', 'hh_30', 'hh_31', 'hh_32', 'hh_33', 'hh_34', 'hh_35', 'hh_36', 'hh_37', 'hh_38', 'hh_39', 'hh_40', 'hh_41', 'hh_42', 'hh_43', 'hh_44', 'hh_45', 'hh_46', 'hh_47']


,LCLid,day,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,hh_10,hh_11,hh_12,hh_13,hh_14,hh_15,hh_16,hh_17,hh_18,hh_19,hh_20,hh_21,hh_22,hh_23,hh_24,hh_25,hh_26,hh_27,hh_28,hh_29,hh_30,hh_31,hh_32,hh_33,hh_34,hh_35,hh_36,hh_37,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
0,MAC000002,2012-10-13,0.2630,0.2690,0.2750,0.2560,0.2110,0.1360,0.1610,0.1190,0.1670,0.1090,0.1680,0.1070,0.1660,0.1170,0.1570,0.1260,0.1460,0.1060,0.1350,0.1910,0.9150,0.9330,0.1220,0.1380,0.0760,0.1330,0.0760,0.1330,0.0850,0.2630,0.1340,0.2350,0.1240,0.1840,0.2300,0.1760,0.3880,0.2600,0.9180,0.2780,0.2670,0.2390,0.2300,0.2330,0.2350,0.1880,0.2590,0.2500
1,MAC000002,2012-10-14,0.2620,0.1660,0.2260,0.0880,0.1260,0.0820,0.1230,0.0830,0.1200,0.0790,0.1210,0.0750,0.1240,0.0730,0.1250,0.0700,0.1300,0.1080,0.1960,0.3460,0.5240,0.0760,0.1290,0.6670,0.2300,0.2200,0.1630,0.0910,0.1700,0.1100,0.1100,0.1210,0.0990,0.1570,0.0930,0.3710,0.3860,1.0850,1.0750,0.9560,0.8210,0.7450,0.7120,0.5110,0.2310,0.2100,0.2780,0.1590
2,MAC000002,2012-10-15,0.1920,0.0970,0.1410,0.0830,0.1320,0.0700,0.1300,0.0740,0.1240,0.0780,0.1180,0.0820,0.1120,0.0870,0.1060,0.1400,0.1200,1.0750,0.1460,0.1230,0.0820,0.1270,0.0770,0.5510,0.1490,0.1290,0.0750,0.1300,0.0750,0.1290,0.0750,0.1280,0.1660,0.1940,0.6950,0.2600,0.2270,0.2550,1.1640,0.2490,0.2250,0.2580,0.2600,0.3340,0.2990,0.2360,0.2410,0.2370
3,MAC000002,2012-10-16,0.2370,0.2370,0.1930,0.1180,0.0980,0.1070,0.0940,0.1090,0.0910,0.1050,0.0910,0.1040,0.0920,0.1030,0.0930,0.1010,0.1440,0.1000,0.4080,0.1020,0.1000,0.1160,0.3540,0.1460,0.1900,0.9910,0.3100,0.1210,0.1130,0.0940,0.1190,0.0870,0.1300,0.2380,0.2040,0.2840,0.4470,0.2660,0.9660,0.1720,0.1920,0.2280,0.2030,0.2110,0.1880,0.2130,0.1570,0.2020
4,MAC000002,2012-10-17,0.1570,0.2110,0.1550,0.1690,0.1010,0.1170,0.0840,0.1180,0.0800,0.1190,0.0750,0.1230,0.0710,0.1260,0.0670,0.1240,0.1180,0.1320,0.3580,0.6280,0.7840,0.6810,0.7490,0.5930,0.5020,0.1150,0.1130,0.0920,0.1240,0.0840,0.1250,0.0780,0.1360,0.2270,0.2070,0.1410,0.2580,0.2170,0.2230,0.0750,0.2300,0.2080,0.2650,0.3770,0.3270,0.2770,0.2880,0.2560



halfhourly_block_0.csv
Ruta: /Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21/halfhourly_dataset/halfhourly_dataset/block_0.csv
Existe: True
Shape muestra: (5, 3)
Columnas: ['LCLid', 'tstp', 'energy(kWh/hh)']


,LCLid,tstp,energy(kWh/hh)
0,MAC000002,2012-10-12 00:30:00.0000000,0
1,MAC000002,2012-10-12 01:00:00.0000000,0
2,MAC000002,2012-10-12 01:30:00.0000000,0
3,MAC000002,2012-10-12 02:00:00.0000000,0
4,MAC000002,2012-10-12 02:30:00.0000000,0


# 4. Información de hogares y variables categóricas

Este archivo contiene el tipo de tarifa y perfil socioeconómico ACORN por hogar. Estas variables serán útiles para interpretar clusters, pero no necesariamente para formar los clusters.

In [4]:
households_path = archivos_principales["informations_households.csv"]
df_households = pd.read_csv(households_path)

print("Shape households:", df_households.shape)
print("Hogares únicos:", df_households["LCLid"].nunique())
display(df_households.head())

print("\nDistribución tipo de tarifa:")
display(df_households["stdorToU"].value_counts(dropna=False).to_frame("hogares"))

print("\nDistribución Acorn_grouped:")
display(df_households["Acorn_grouped"].value_counts(dropna=False).to_frame("hogares"))

print("\nCruce tarifa vs Acorn_grouped:")
display(pd.crosstab(df_households["Acorn_grouped"], df_households["stdorToU"]))

Shape households: (5566, 5)
Hogares únicos: 5566


,LCLid,stdorToU,Acorn,Acorn_grouped,file
0,MAC005492,ToU,ACORN-,ACORN-,block_0
1,MAC001074,ToU,ACORN-,ACORN-,block_0
2,MAC000002,Std,ACORN-A,Affluent,block_0
3,MAC003613,Std,ACORN-A,Affluent,block_0
4,MAC003597,Std,ACORN-A,Affluent,block_0



Distribución tipo de tarifa:


,hogares
stdorToU,
Std,4443
ToU,1123



Distribución Acorn_grouped:


,hogares
Acorn_grouped,
Affluent,2192
Adversity,1816
Comfortable,1507
ACORN-U,49
ACORN-,2



Cruce tarifa vs Acorn_grouped:


stdorToU,Std,ToU
Acorn_grouped,,
ACORN-,0,2
ACORN-U,39,10
Adversity,1518,298
Affluent,1702,490
Comfortable,1184,323


# 5. Consumo diario consolidado

El archivo `daily_dataset.csv` es la base más conveniente para construir perfiles por hogar porque ya resume consumo por día. Se revisan cobertura temporal, hogares y calidad básica.

In [5]:
daily_path = archivos_principales["daily_dataset.csv"]

# Carga completa del daily_dataset. Es grande pero manejable en esta máquina (~335 MB).
df_daily = pd.read_csv(daily_path, parse_dates=["day"])

print("Shape daily:", df_daily.shape)
print("Hogares únicos:", df_daily["LCLid"].nunique())
print("Rango de fechas:", df_daily["day"].min(), "a", df_daily["day"].max())
print("Duplicados:", df_daily.duplicated().sum())

display(df_daily.head())
display(df_daily.describe().T)

print("\nValores faltantes:")
display(df_daily.isna().sum().to_frame("nulos"))

Shape daily: (3510433, 9)
Hogares únicos: 5566
Rango de fechas: 2011-11-23 00:00:00 a 2014-02-28 00:00:00
Duplicados: 0


,LCLid,day,energy_median,energy_mean,energy_max,energy_count,energy_std,energy_sum,energy_min
0,MAC000131,2011-12-15,0.4850,0.4320,0.8680,22,0.2391,9.5050,0.0720
1,MAC000131,2011-12-16,0.1415,0.2962,1.1160,48,0.2815,14.2160,0.0310
2,MAC000131,2011-12-17,0.1015,0.1898,0.6850,48,0.1884,9.1110,0.0640
3,MAC000131,2011-12-18,0.1140,0.2190,0.6760,48,0.2029,10.5110,0.0650
4,MAC000131,2011-12-19,0.1910,0.3260,0.7880,48,0.2592,15.6470,0.0660


,count,mean,min,25%,50%,75%,max,std
day,3510433,2013-03-27 21:09:43.463407360,2011-11-23 00:00:00,2012-10-21 00:00:00,2013-03-30 00:00:00,2013-09-10 00:00:00,2014-02-28 00:00:00,NaN
energy_median,"3,510,403.0000",0.1587,0.0000,0.0670,0.1145,0.1910,6.9705,0.1702
energy_mean,"3,510,403.0000",0.2117,0.0000,0.0981,0.1633,0.2625,6.9283,0.1908
energy_max,"3,510,403.0000",0.8345,0.0000,0.3460,0.6880,1.1280,10.7610,0.6683
energy_count,"3,510,433.0000",47.8036,0.0000,48.0000,48.0000,48.0000,48.0000,2.8110
energy_std,"3,499,102.0000",0.1727,0.0000,0.0691,0.1328,0.2293,4.0246,0.1531
energy_sum,"3,510,403.0000",10.1241,0.0000,4.6820,7.8150,12.5690,332.5560,9.1288
energy_min,"3,510,403.0000",0.0596,0.0000,0.0200,0.0390,0.0710,6.5240,0.0870



Valores faltantes:


,nulos
LCLid,0
day,0
energy_median,30
energy_mean,30
energy_max,30
energy_count,0
energy_std,11331
energy_sum,30
energy_min,30


In [6]:
daily_coverage = (
    df_daily
    .groupby("LCLid")
    .agg(
        dias_registrados=("day", "nunique"),
        fecha_min=("day", "min"),
        fecha_max=("day", "max"),
        consumo_total=("energy_sum", "sum"),
        consumo_medio_diario=("energy_sum", "mean"),
        consumo_max_dia=("energy_sum", "max"),
        consumo_std_diario=("energy_sum", "std")
    )
    .reset_index()
)

print("Shape cobertura por hogar:", daily_coverage.shape)
display(daily_coverage.describe().T)

plt.figure(figsize=(9, 5))
plt.hist(daily_coverage["dias_registrados"], bins=40)
plt.title("Distribución de días registrados por hogar")
plt.xlabel("Días registrados")
plt.ylabel("Número de hogares")
plt.grid(True)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "daily_dias_registrados_por_hogar.png", dpi=300, bbox_inches="tight")
plt.show()

Shape cobertura por hogar: (5566, 8)


,count,mean,min,25%,50%,75%,max,std
dias_registrados,"5,566.0000",630.6922,1.0000,600.0000,651.0000,684.0000,829.0000,111.9451
fecha_min,5566,2012-05-17 06:51:05.756377856,2011-11-23 00:00:00,2012-04-11 00:00:00,2012-05-15 00:00:00,2012-06-23 00:00:00,2013-10-29 00:00:00,NaN
fecha_max,5566,2014-02-06 17:05:32.446999808,2012-11-06 00:00:00,2014-02-28 00:00:00,2014-02-28 00:00:00,2014-02-28 00:00:00,2014-02-28 00:00:00,NaN
consumo_total,"5,566.0000","6,385.1641",0.0000,"3,287.4188","5,216.7640","8,110.7730","65,623.6490","4,759.2646"
consumo_medio_diario,"5,561.0000",10.1757,0.0000,5.3631,8.3078,12.7945,101.1150,7.4653
consumo_max_dia,"5,561.0000",27.0382,0.0000,13.3590,21.3150,33.6130,332.5560,22.2727
consumo_std_diario,"5,561.0000",3.8819,0.0000,1.6754,2.7569,4.5017,61.4549,3.9132


In [7]:
df_daily["mes"] = df_daily["day"].dt.to_period("M").astype(str)
consumo_mensual = (
    df_daily
    .groupby("mes")
    .agg(
        consumo_total=("energy_sum", "sum"),
        consumo_medio_diario=("energy_sum", "mean"),
        hogares=("LCLid", "nunique")
    )
    .reset_index()
)

plt.figure(figsize=(12, 5))
plt.plot(consumo_mensual["mes"], consumo_mensual["consumo_medio_diario"], marker="o")
plt.title("Consumo medio diario por mes")
plt.xlabel("Mes")
plt.ylabel("Consumo medio diario")
plt.xticks(rotation=60, ha="right")
plt.grid(True)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "daily_consumo_medio_por_mes.png", dpi=300, bbox_inches="tight")
plt.show()

display(consumo_mensual.head())

,mes,consumo_total,consumo_medio_diario,hogares
0,2011-11,"3,228.4790",9.3309,76
1,2011-12,"100,645.9270",12.3084,411
2,2012-01,"187,743.2940",12.9042,581
3,2012-02,"280,329.0250",13.3815,843
4,2012-03,"354,699.7670",11.0447,1235


# 6. Clima diario

El clima puede ayudar a interpretar estacionalidad y consumo. Se revisa rango temporal, variables principales y posibles columnas para unir por fecha.

In [8]:
weather_path = archivos_principales["weather_daily_darksky.csv"]
df_weather = pd.read_csv(weather_path, parse_dates=["time"])
df_weather["day"] = df_weather["time"].dt.normalize()

weather_duplicate_days = df_weather["day"].duplicated().sum()
df_weather_daily = (
    df_weather
    .sort_values("time")
    .drop_duplicates("day", keep="first")
    .copy()
)

print("Shape weather original:", df_weather.shape)
print("Shape weather deduplicado por día:", df_weather_daily.shape)
print("Días duplicados en weather:", weather_duplicate_days)
print("Rango de fechas:", df_weather_daily["day"].min(), "a", df_weather_daily["day"].max())
display(df_weather_daily.head())

weather_cols = [
    "day", "temperatureMax", "temperatureMin", "temperatureHigh",
    "temperatureLow", "humidity", "windSpeed", "cloudCover", "pressure", "uvIndex"
]
weather_cols = [col for col in weather_cols if col in df_weather_daily.columns]
display(df_weather_daily[weather_cols].describe().T)

Shape weather original: (882, 33)
Shape weather deduplicado por día: (879, 33)
Días duplicados en weather: 3
Rango de fechas: 2011-11-01 00:00:00 a 2014-03-30 00:00:00


,temperatureMax,temperatureMaxTime,windBearing,icon,dewPoint,temperatureMinTime,cloudCover,windSpeed,pressure,apparentTemperatureMinTime,apparentTemperatureHigh,precipType,visibility,humidity,apparentTemperatureHighTime,apparentTemperatureLow,apparentTemperatureMax,uvIndex,time,sunsetTime,temperatureLow,temperatureMin,temperatureHigh,sunriseTime,temperatureHighTime,uvIndexTime,summary,temperatureLowTime,apparentTemperatureMin,apparentTemperatureMaxTime,apparentTemperatureLowTime,moonPhase,day
13,15.5700,2011-11-01 15:00:00,208,partly-cloudy-day,10.1300,2011-11-01 22:00:00,0.3600,2.4500,"1,009.4600",2011-11-01 22:00:00,15.5700,rain,12.6800,0.8400,2011-11-01 15:00:00,7.3300,15.5700,1.0000,2011-11-01,2011-11-01 16:36:03,8.8800,9.6800,15.5700,2011-11-01 06:54:29,2011-11-01 15:00:00,2011-11-01 10:00:00,Partly cloudy until evening.,2011-11-02 03:00:00,9.0100,2011-11-01 15:00:00,2011-11-02 03:00:00,0.2100,2011-11-01
60,15.1900,2011-11-02 23:00:00,134,partly-cloudy-night,10.2300,2011-11-02 03:00:00,0.3900,4.3900,"1,004.7900",2011-11-02 03:00:00,15.0600,rain,11.8300,0.8700,2011-11-02 13:00:00,13.9900,15.1900,1.0000,2011-11-02,2011-11-02 16:34:15,13.9900,8.8800,15.0600,2011-11-02 06:56:16,2011-11-02 13:00:00,2011-11-02 09:00:00,Partly cloudy throughout the day.,2011-11-02 19:00:00,7.3300,2011-11-02 23:00:00,2011-11-02 19:00:00,0.2400,2011-11-02
34,17.4100,2011-11-03 14:00:00,154,partly-cloudy-day,13.3900,2011-11-03 21:00:00,0.5200,3.9900,993.4000,2011-11-03 21:00:00,17.4100,rain,12.1500,0.8900,2011-11-03 14:00:00,12.5200,17.4100,1.0000,2011-11-03,2011-11-03 16:32:29,12.5200,12.7900,17.4100,2011-11-03 06:58:03,2011-11-03 14:00:00,2011-11-03 10:00:00,Partly cloudy throughout the day.,2011-11-04 07:00:00,12.7900,2011-11-03 14:00:00,2011-11-04 07:00:00,0.2700,2011-11-03
31,15.5400,2011-11-04 11:00:00,179,fog,12.0300,2011-11-04 23:00:00,0.5000,2.6200,995.5400,2011-11-04 23:00:00,15.5400,rain,10.6900,0.9100,2011-11-04 11:00:00,10.1700,15.5400,1.0000,2011-11-04,2011-11-04 16:30:44,10.1700,11.5300,15.5400,2011-11-04 06:59:49,2011-11-04 11:00:00,2011-11-04 10:00:00,Foggy overnight.,2011-11-05 02:00:00,11.5300,2011-11-04 11:00:00,2011-11-05 02:00:00,0.3100,2011-11-04
46,13.9400,2011-11-05 15:00:00,346,fog,10.9600,2011-11-05 02:00:00,0.6500,2.7000,"1,007.3900",2011-11-05 02:00:00,13.9400,rain,4.6000,0.9200,2011-11-05 15:00:00,7.0300,13.9400,1.0000,2011-11-05,2011-11-05 16:29:01,9.4600,10.1700,13.9400,2011-11-05 07:01:36,2011-11-05 15:00:00,2011-11-05 10:00:00,Foggy in the morning.,2011-11-06 05:00:00,10.1700,2011-11-05 15:00:00,2011-11-06 06:00:00,0.3400,2011-11-05


,count,mean,min,25%,50%,75%,max,std
day,879,2013-01-13 18:19:14.948805632,2011-11-01 00:00:00,2012-06-07 12:00:00,2013-01-14 00:00:00,2013-08-21 12:00:00,2014-03-30 00:00:00,NaN
temperatureMax,879.0000,13.6607,-0.0600,9.5150,12.6200,17.9200,32.4000,6.1852
temperatureMin,879.0000,7.4214,-5.6400,3.7100,7.1000,11.3000,20.5400,4.8896
temperatureHigh,879.0000,13.5426,-0.8100,9.2150,12.4600,17.9000,32.4000,6.2629
temperatureLow,879.0000,7.7212,-5.6400,4.0000,7.5400,11.4700,20.5400,4.8718
humidity,879.0000,0.7822,0.4300,0.7200,0.7900,0.8600,0.9800,0.0953
windSpeed,879.0000,3.5823,0.2000,2.3700,3.4400,4.5750,9.9600,1.6946
cloudCover,878.0000,0.4781,0.0000,0.3500,0.4700,0.6000,1.0000,0.1934
pressure,879.0000,"1,014.1070",979.2500,"1,007.4250","1,014.6200","1,021.7500","1,040.9200",11.0664
uvIndex,878.0000,2.5410,0.0000,1.0000,2.0000,4.0000,7.0000,1.8359


# 7. ACORN details

El archivo ACORN describe características demográficas por categoría. Es útil para narrativa e interpretación, no como variable directa de entrenamiento.

In [9]:
acorn_path = archivos_principales["acorn_details.csv"]
df_acorn = pd.read_csv(acorn_path, encoding="latin1")

print("Shape acorn_details:", df_acorn.shape)
display(df_acorn.head(10))

print("Categorías principales:")
display(df_acorn["MAIN CATEGORIES"].value_counts().to_frame("filas"))

Shape acorn_details: (826, 20)


,MAIN CATEGORIES,CATEGORIES,REFERENCE,ACORN-A,ACORN-B,ACORN-C,ACORN-D,ACORN-E,ACORN-F,ACORN-G,ACORN-H,ACORN-I,ACORN-J,ACORN-K,ACORN-L,ACORN-M,ACORN-N,ACORN-O,ACORN-P,ACORN-Q
0,POPULATION,Age,Age 0-4,77.0000,83.0000,72.0000,100.0000,120.0000,77.0000,97.0000,97.0000,63.0000,119.0000,67.0000,114.0000,113.0000,89.0000,123.0000,138.0000,133.0000
1,POPULATION,Age,Age 5-17,117.0000,109.0000,87.0000,69.0000,94.0000,95.0000,102.0000,106.0000,67.0000,95.0000,64.0000,108.0000,116.0000,86.0000,89.0000,136.0000,106.0000
2,POPULATION,Age,Age 18-24,64.0000,73.0000,67.0000,107.0000,100.0000,71.0000,83.0000,89.0000,62.0000,104.0000,459.0000,97.0000,96.0000,86.0000,117.0000,109.0000,110.0000
3,POPULATION,Age,Age 25-34,52.0000,63.0000,62.0000,197.0000,151.0000,66.0000,90.0000,88.0000,63.0000,132.0000,145.0000,109.0000,96.0000,90.0000,140.0000,120.0000,120.0000
4,POPULATION,Age,Age 35-49,102.0000,105.0000,91.0000,124.0000,118.0000,93.0000,102.0000,103.0000,76.0000,111.0000,67.0000,99.0000,98.0000,90.0000,102.0000,103.0000,100.0000
5,POPULATION,Age,Age 50-64,124.0000,121.0000,120.0000,72.0000,82.0000,126.0000,109.0000,107.0000,112.0000,90.0000,41.0000,95.0000,96.0000,103.0000,89.0000,78.0000,89.0000
6,POPULATION,Age,Aged 65-74,125.0000,120.0000,152.0000,55.0000,61.0000,144.0000,108.0000,104.0000,182.0000,72.0000,29.0000,91.0000,93.0000,125.0000,73.0000,59.0000,76.0000
7,POPULATION,Age,Aged 75 plus,112.0000,103.0000,157.0000,49.0000,57.0000,117.0000,98.0000,96.0000,220.0000,66.0000,32.0000,87.0000,96.0000,152.0000,72.0000,56.0000,76.0000
8,POPULATION,Geography,England,107.0000,101.0000,103.0000,114.0000,106.0000,75.0000,107.0000,106.0000,102.0000,106.0000,95.0000,93.0000,97.0000,89.0000,97.0000,110.0000,97.0000
9,POPULATION,Geography,Northern Ireland,30.0000,95.0000,45.0000,2.0000,49.0000,462.0000,53.0000,104.0000,30.0000,91.0000,56.0000,87.0000,131.0000,67.0000,95.0000,75.0000,43.0000


Categorías principales:


,filas
MAIN CATEGORIES,
DIGITAL,372
FINANCE,94
LEISURE TIME,92
SHOPPING,44
POPULATION,35
ECONOMY,34
MARKETING CHANNELS,27
TRANSPORT,25
HOUSING,24


# 8. Bloques horarios hhblock

`hhblock_dataset` contiene 48 columnas por día, una por cada media hora. Esto permite estudiar comportamiento intradía sin cargar el dataset halfhourly raw completo. Se inspecciona una muestra del primer bloque.

In [10]:
hhblock_path = carpetas_bloques["hhblock_blocks"] / "block_0.csv"
df_hhblock_0 = pd.read_csv(hhblock_path)

hh_cols = [col for col in df_hhblock_0.columns if col.startswith("hh_")]
print("Shape hhblock block_0:", df_hhblock_0.shape)
print("Número de columnas de media hora:", len(hh_cols))
print("Hogares únicos en block_0:", df_hhblock_0["LCLid"].nunique())
print("Rango de fechas:", df_hhblock_0["day"].min(), "a", df_hhblock_0["day"].max())
display(df_hhblock_0.head())

perfil_intradia = (
    df_hhblock_0[hh_cols]
    .apply(pd.to_numeric, errors="coerce")
    .mean()
    .reset_index()
)
perfil_intradia.columns = ["periodo", "consumo_promedio"]
perfil_intradia["media_hora"] = perfil_intradia["periodo"].str.replace("hh_", "", regex=False).astype(int)
perfil_intradia = perfil_intradia.sort_values("media_hora")

plt.figure(figsize=(10, 5))
plt.plot(perfil_intradia["media_hora"], perfil_intradia["consumo_promedio"], marker="o")
plt.title("Perfil intradía promedio - hhblock block_0")
plt.xlabel("Periodo de media hora")
plt.ylabel("Consumo promedio")
plt.grid(True)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "hhblock_block0_perfil_intradia.png", dpi=300, bbox_inches="tight")
plt.show()

display(perfil_intradia.head())

Shape hhblock block_0: (25286, 50)
Número de columnas de media hora: 48
Hogares únicos en block_0: 50
Rango de fechas: 2011-12-04 a 2014-02-27


,LCLid,day,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,hh_10,hh_11,hh_12,hh_13,hh_14,hh_15,hh_16,hh_17,hh_18,hh_19,hh_20,hh_21,hh_22,hh_23,hh_24,hh_25,hh_26,hh_27,hh_28,hh_29,hh_30,hh_31,hh_32,hh_33,hh_34,hh_35,hh_36,hh_37,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
0,MAC000002,2012-10-13,0.2630,0.2690,0.2750,0.2560,0.2110,0.1360,0.1610,0.1190,0.1670,0.1090,0.1680,0.1070,0.1660,0.1170,0.1570,0.1260,0.1460,0.1060,0.1350,0.1910,0.9150,0.9330,0.1220,0.1380,0.0760,0.1330,0.0760,0.1330,0.0850,0.2630,0.1340,0.2350,0.1240,0.1840,0.2300,0.1760,0.3880,0.2600,0.9180,0.2780,0.2670,0.2390,0.2300,0.2330,0.2350,0.1880,0.2590,0.2500
1,MAC000002,2012-10-14,0.2620,0.1660,0.2260,0.0880,0.1260,0.0820,0.1230,0.0830,0.1200,0.0790,0.1210,0.0750,0.1240,0.0730,0.1250,0.0700,0.1300,0.1080,0.1960,0.3460,0.5240,0.0760,0.1290,0.6670,0.2300,0.2200,0.1630,0.0910,0.1700,0.1100,0.1100,0.1210,0.0990,0.1570,0.0930,0.3710,0.3860,1.0850,1.0750,0.9560,0.8210,0.7450,0.7120,0.5110,0.2310,0.2100,0.2780,0.1590
2,MAC000002,2012-10-15,0.1920,0.0970,0.1410,0.0830,0.1320,0.0700,0.1300,0.0740,0.1240,0.0780,0.1180,0.0820,0.1120,0.0870,0.1060,0.1400,0.1200,1.0750,0.1460,0.1230,0.0820,0.1270,0.0770,0.5510,0.1490,0.1290,0.0750,0.1300,0.0750,0.1290,0.0750,0.1280,0.1660,0.1940,0.6950,0.2600,0.2270,0.2550,1.1640,0.2490,0.2250,0.2580,0.2600,0.3340,0.2990,0.2360,0.2410,0.2370
3,MAC000002,2012-10-16,0.2370,0.2370,0.1930,0.1180,0.0980,0.1070,0.0940,0.1090,0.0910,0.1050,0.0910,0.1040,0.0920,0.1030,0.0930,0.1010,0.1440,0.1000,0.4080,0.1020,0.1000,0.1160,0.3540,0.1460,0.1900,0.9910,0.3100,0.1210,0.1130,0.0940,0.1190,0.0870,0.1300,0.2380,0.2040,0.2840,0.4470,0.2660,0.9660,0.1720,0.1920,0.2280,0.2030,0.2110,0.1880,0.2130,0.1570,0.2020
4,MAC000002,2012-10-17,0.1570,0.2110,0.1550,0.1690,0.1010,0.1170,0.0840,0.1180,0.0800,0.1190,0.0750,0.1230,0.0710,0.1260,0.0670,0.1240,0.1180,0.1320,0.3580,0.6280,0.7840,0.6810,0.7490,0.5930,0.5020,0.1150,0.1130,0.0920,0.1240,0.0840,0.1250,0.0780,0.1360,0.2270,0.2070,0.1410,0.2580,0.2170,0.2230,0.0750,0.2300,0.2080,0.2650,0.3770,0.3270,0.2770,0.2880,0.2560


,periodo,consumo_promedio,media_hora
0,hh_0,0.3526,0
1,hh_1,0.3158,1
2,hh_2,0.2919,2
3,hh_3,0.2769,3
4,hh_4,0.2667,4


# 9. Halfhourly raw: inspección mínima

El dataset `halfhourly_dataset` está en formato largo y es muy pesado. Se revisa solo una muestra para entender su estructura. Para análisis completo se recomienda procesarlo por chunks o por bloques seleccionados.

In [11]:
halfhourly_path = carpetas_bloques["halfhourly_blocks"] / "block_0.csv"
df_halfhourly_sample = pd.read_csv(halfhourly_path, nrows=100_000)

print("Muestra halfhourly:", df_halfhourly_sample.shape)
print("Columnas:", df_halfhourly_sample.columns.tolist())
print("Hogares únicos en muestra:", df_halfhourly_sample["LCLid"].nunique())
display(df_halfhourly_sample.head())

energy_col = "energy(kWh/hh)"
df_halfhourly_sample[energy_col] = pd.to_numeric(df_halfhourly_sample[energy_col], errors="coerce")
display(df_halfhourly_sample[energy_col].describe().to_frame("energia_muestra"))

Muestra halfhourly: (100000, 3)
Columnas: ['LCLid', 'tstp', 'energy(kWh/hh)']
Hogares únicos en muestra: 5


,LCLid,tstp,energy(kWh/hh)
0,MAC000002,2012-10-12 00:30:00.0000000,0
1,MAC000002,2012-10-12 01:00:00.0000000,0
2,MAC000002,2012-10-12 01:30:00.0000000,0
3,MAC000002,2012-10-12 02:00:00.0000000,0
4,MAC000002,2012-10-12 02:30:00.0000000,0


,energia_muestra
count,"99,995.0000"
mean,0.5802
std,0.6805
min,0.0000
25%,0.1100
50%,0.2720
75%,0.8730
max,5.2500


# 10. Validación de joins principales

Se prueba la relación entre consumo diario, hogares y clima. Esto anticipa la construcción de una tabla model-ready para clustering.

In [12]:
df_daily_join = df_daily.merge(
    df_households[["LCLid", "stdorToU", "Acorn", "Acorn_grouped"]],
    on="LCLid",
    how="left"
)

df_daily_join = df_daily_join.merge(
    df_weather_daily[["day", "temperatureMax", "temperatureMin", "humidity", "cloudCover", "windSpeed"]],
    on="day",
    how="left"
)

print("Shape daily original:", df_daily.shape)
print("Shape daily + households + weather:", df_daily_join.shape)
print("Hogares sin metadata:", df_daily_join["stdorToU"].isna().sum())
print("Registros sin clima:", df_daily_join["temperatureMax"].isna().sum())
display(df_daily_join.head())

join_quality = pd.DataFrame({
    "columna": df_daily_join.columns,
    "nulos": df_daily_join.isna().sum().values,
    "pct_nulos": (df_daily_join.isna().mean().values * 100).round(2)
})
display(join_quality.sort_values("pct_nulos", ascending=False).head(15))

Shape daily original: (3510433, 10)
Shape daily + households + weather: (3510433, 18)
Hogares sin metadata: 0
Registros sin clima: 10714


,LCLid,day,energy_median,energy_mean,energy_max,energy_count,energy_std,energy_sum,energy_min,mes,stdorToU,Acorn,Acorn_grouped,temperatureMax,temperatureMin,humidity,cloudCover,windSpeed
0,MAC000131,2011-12-15,0.4850,0.4320,0.8680,22,0.2391,9.5050,0.0720,2011-12,Std,ACORN-E,Affluent,7.9700,4.0800,0.7700,0.4200,4.7100
1,MAC000131,2011-12-16,0.1415,0.2962,1.1160,48,0.2815,14.2160,0.0310,2011-12,Std,ACORN-E,Affluent,4.6800,1.8000,0.8800,0.7000,3.7100
2,MAC000131,2011-12-17,0.1015,0.1898,0.6850,48,0.1884,9.1110,0.0640,2011-12,Std,ACORN-E,Affluent,5.3500,0.2400,0.8600,0.3700,3.9900
3,MAC000131,2011-12-18,0.1140,0.2190,0.6760,48,0.2029,10.5110,0.0650,2011-12,Std,ACORN-E,Affluent,5.4900,-0.5600,0.8400,0.2200,3.6000
4,MAC000131,2011-12-19,0.1910,0.3260,0.7880,48,0.2592,15.6470,0.0660,2011-12,Std,ACORN-E,Affluent,6.6400,-0.8400,0.9400,0.4700,2.7000


,columna,nulos,pct_nulos
16,cloudCover,15813,0.4500
6,energy_std,11331,0.3200
17,windSpeed,10714,0.3100
15,humidity,10714,0.3100
14,temperatureMin,10714,0.3100
13,temperatureMax,10714,0.3100
10,stdorToU,0,0.0000
12,Acorn_grouped,0,0.0000
11,Acorn,0,0.0000
0,LCLid,0,0.0000


# 11. Primer perfil agregado por hogar desde originales

Se construye una primera tabla agregada por hogar desde `daily_dataset.csv`. Esta tabla todavía no es el modelo final, pero sirve como base para el siguiente notebook de clustering.

In [13]:
def asignar_estacion(mes):
    if mes in [12, 1, 2]:
        return "invierno"
    if mes in [3, 4, 5]:
        return "primavera"
    if mes in [6, 7, 8]:
        return "verano"
    return "otono"

df_daily_join["dia_semana"] = df_daily_join["day"].dt.dayofweek
df_daily_join["es_finde"] = df_daily_join["dia_semana"] >= 5
df_daily_join["mes_num"] = df_daily_join["day"].dt.month
df_daily_join["estacion"] = df_daily_join["mes_num"].apply(asignar_estacion)

perfil_hogar_original = (
    df_daily_join
    .groupby("LCLid")
    .agg(
        consumo_total=("energy_sum", "sum"),
        consumo_medio=("energy_sum", "mean"),
        consumo_std=("energy_sum", "std"),
        consumo_max=("energy_sum", "max"),
        consumo_min=("energy_sum", "min"),
        dias_registrados=("day", "nunique"),
        tarifa=("stdorToU", "first"),
        acorn=("Acorn", "first"),
        Acorn_grouped=("Acorn_grouped", "first")
    )
    .reset_index()
)

consumo_estacional = df_daily_join.pivot_table(
    index="LCLid",
    columns="estacion",
    values="energy_sum",
    aggfunc="mean"
).rename(columns={
    "invierno": "consumo_invierno",
    "primavera": "consumo_primavera",
    "verano": "consumo_verano",
    "otono": "consumo_otono"
})

consumo_finde = df_daily_join[df_daily_join["es_finde"]].groupby("LCLid")["energy_sum"].mean().rename("consumo_finde")
consumo_semana = df_daily_join[~df_daily_join["es_finde"]].groupby("LCLid")["energy_sum"].mean().rename("consumo_semana")

perfil_hogar_original = (
    perfil_hogar_original
    .merge(consumo_estacional.reset_index(), on="LCLid", how="left")
    .merge(consumo_finde.reset_index(), on="LCLid", how="left")
    .merge(consumo_semana.reset_index(), on="LCLid", how="left")
)

print("Shape perfil_hogar_original:", perfil_hogar_original.shape)
display(perfil_hogar_original.head())
display(perfil_hogar_original.describe(include="all").T)

Shape perfil_hogar_original: (5566, 16)


,LCLid,consumo_total,consumo_medio,consumo_std,consumo_max,consumo_min,dias_registrados,tarifa,acorn,Acorn_grouped,consumo_invierno,consumo_otono,consumo_primavera,consumo_verano,consumo_finde,consumo_semana
0,MAC000002,"6,095.6720",12.0706,4.4945,39.2840,0.1860,505,Std,ACORN-A,Affluent,13.2038,12.2826,12.4583,9.1410,12.8539,11.7582
1,MAC000003,"14,080.8620",19.0282,11.6061,50.7590,0.0700,740,Std,ACORN-P,Adversity,27.5882,17.2397,18.7572,12.2292,18.7746,19.1287
2,MAC000004,"1,119.8390",1.6916,0.4314,7.3540,0.0000,662,Std,ACORN-E,Affluent,1.8376,1.7252,1.6804,1.5225,1.7367,1.6737
3,MAC000005,"2,911.0060",4.5627,1.4705,16.7080,0.0320,638,ToU,ACORN-C,Affluent,5.5373,4.4996,4.4816,3.7122,4.5833,4.5545
4,MAC000006,"2,167.4480",2.8482,0.8822,6.6360,0.0010,761,Std,ACORN-Q,Adversity,3.0004,2.9739,2.9350,2.4623,2.7354,2.8928


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
LCLid,5566,5566,MAC000002,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
consumo_total,"5,566.0000",NaN,NaN,NaN,"6,385.1641","4,759.2646",0.0000,"3,287.4188","5,216.7640","8,110.7730","65,623.6490"
consumo_medio,"5,561.0000",NaN,NaN,NaN,10.1757,7.4653,0.0000,5.3631,8.3078,12.7945,101.1150
consumo_std,"5,561.0000",NaN,NaN,NaN,3.8819,3.9132,0.0000,1.6754,2.7569,4.5017,61.4549
consumo_max,"5,561.0000",NaN,NaN,NaN,27.0382,22.2727,0.0000,13.3590,21.3150,33.6130,332.5560
consumo_min,"5,561.0000",NaN,NaN,NaN,0.1955,0.6035,0.0000,0.0410,0.0840,0.1700,18.4110
dias_registrados,"5,566.0000",NaN,NaN,NaN,630.6922,111.9451,1.0000,600.0000,651.0000,684.0000,829.0000
tarifa,5566,2,Std,4443,NaN,NaN,NaN,NaN,NaN,NaN,NaN
acorn,5566,19,ACORN-E,1567,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Acorn_grouped,5566,5,Affluent,2192,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
perfil_hogar_original.to_csv(OUTPUT_DIR / "perfil_hogar_desde_originales_inspeccion.csv", index=False)
df_inventario.to_csv(OUTPUT_DIR / "inventario_datasets_originales.csv", index=False)
daily_coverage.to_csv(OUTPUT_DIR / "daily_cobertura_por_hogar.csv", index=False)

print("Archivos de inspección guardados en:", OUTPUT_DIR)

Archivos de inspección guardados en: /Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/output/inspeccion_originales


# 12. Observaciones iniciales y siguientes pasos

Observaciones iniciales esperadas:

- `daily_dataset.csv` es la mejor base para construir perfiles de consumo por hogar sin cargar la granularidad completa de media hora.
- `informations_households.csv` aporta variables categóricas de negocio: tipo de tarifa y ACORN.
- `weather_daily_darksky.csv` puede ayudar a explicar estacionalidad y sensibilidad climática.
- `hhblock_dataset` permite estudiar horarios pico/valle con 48 periodos por día.
- `halfhourly_dataset` debe tratarse por bloques o muestras por su tamaño.

Siguientes pasos recomendados:

1. Definir variables de comportamiento por hogar usando `daily_dataset.csv`.
2. Incorporar features climáticas agregadas por hogar o por periodo.
3. Usar `hhblock_dataset` para enriquecer patrones horarios, por ejemplo consumo nocturno, mañana, tarde y noche.
4. Construir una nueva libreta de clustering con variables de magnitud, variabilidad, estacionalidad y horario.
5. Usar `stdorToU` y `Acorn_grouped` para interpretar los clusters, no para entrenarlos directamente.